#### Load Data

In [14]:
import json

selfcheck_scores_unigram = {}
selfcheck_scores_bigram = {}
selfcheck_scores_trigram = {}
selfcheck_scores_fourgram = {}
selfcheck_scores_fivegram = {}

with open('data/selfcheck_scores_unigram.json', 'r') as file:
    selfcheck_scores_unigram = json.load(file)
with open('data/selfcheck_scores_bigram.json', 'r') as file:
    selfcheck_scores_bigram = json.load(file)
with open('data/selfcheck_scores_trigram.json', 'r') as file:
    selfcheck_scores_trigram = json.load(file)
with open('data/selfcheck_scores_fourgram.json', 'r') as file:
    selfcheck_scores_fourgram = json.load(file)
with open('data/selfcheck_scores_fivegram.json', 'r') as file:
    selfcheck_scores_fivegram = json.load(file)

In [15]:
def process_ngram_score(scores):
    scores_sent_avg_final = {}
    scores_sent_max_final = {}
    scores_doc_avg_final = {}
    scores_doc_avg_max_final = {}

    for idx in scores:
        scores_sent = scores[idx]['sent_level']
        scores_doc = scores[idx]['doc_level']

        scores_sent_avg = scores_sent['avg_neg_logprob']
        scores_sent_avg = [score[0] for score in scores_sent_avg]

        scores_sent_max =  scores_sent['max_neg_logprob']
        scores_sent_max = [score[0] for score in scores_sent_max]

        score_doc_avg = scores_doc['avg_neg_logprob']
        score_doc_abg_max = scores_doc['avg_max_neg_logprob']

        scores_sent_avg_final[int(idx)] = scores_sent_avg
        scores_sent_max_final[int(idx)] = scores_sent_max
        scores_doc_avg_final[int(idx)] = score_doc_avg
        scores_doc_avg_max_final[int(idx)] = score_doc_abg_max
    return [scores_sent_avg_final, scores_sent_max_final, scores_doc_avg_final, scores_doc_avg_max_final]
    
scores_unigram = process_ngram_score(selfcheck_scores_unigram)
scores_bigram = process_ngram_score(selfcheck_scores_bigram)
scores_trigram = process_ngram_score(selfcheck_scores_trigram)
scores_fourgram = process_ngram_score(selfcheck_scores_fourgram)
scores_fivegram = process_ngram_score(selfcheck_scores_fivegram)

In [16]:
import json
import numpy as np

with open("data/dataset_v3.json", "r") as f:
    content = f.read()
dataset = json.loads(content)
print("The lenght of the dataset: {}".format(len(dataset)))
print(dataset[0].keys())


label_mapping = {
    'accurate': 0.0,
    'minor_inaccurate': 0.5,
    'major_inaccurate': 1.0,
}

human_label_detect_False   = {}
human_label_detect_True    = {}
human_label_detect_False_h = {}

for i_ in range(len(dataset)):
    dataset_i = dataset[i_]
    idx = dataset_i["wiki_bio_test_idx"]
    
    raw_label = np.array([label_mapping[x] for x in dataset_i['annotation']])
    human_label_detect_False[idx] = (raw_label > 0.499).astype(np.int32).tolist()
    human_label_detect_True[idx] = (raw_label < 0.499).astype(np.int32).tolist()
    average_score = np.mean(raw_label)
    if (average_score < 0.99):
        human_label_detect_False_h[idx] = (raw_label > 0.99).astype(np.int32).tolist()
print("Length of False:", len(human_label_detect_False))
print("Length of True:", len(human_label_detect_True)) 
print("Length of False_h:", len(human_label_detect_False_h))

indices = [x['wiki_bio_test_idx'] for x in dataset] 

The lenght of the dataset: 238
dict_keys(['gpt3_text', 'wiki_bio_text', 'gpt3_sentences', 'annotation', 'wiki_bio_test_idx', 'gpt3_text_samples'])
Length of False: 238
Length of True: 238
Length of False_h: 206


In [17]:
from sklearn.metrics import precision_recall_curve, auc

def unroll_pred(scores, indices):
    unrolled = []
    
    for idx in indices:
        unrolled.extend(scores[idx])
    return unrolled

def get_PR_with_human_labels(preds, human_labels, pos_label=1, oneminus_pred=False):
    indices = [k for k in human_labels.keys()]
    unroll_preds = unroll_pred(preds, indices)
    if oneminus_pred:
        unroll_preds = [1.0-x for x in unroll_preds]
    unroll_labels = unroll_pred(human_labels, indices)
    
    assert(len(unroll_preds) == len(unroll_labels))

    # print("Length: ", len(unroll_preds))
    p, r, threshold = precision_recall_curve(unroll_labels, unroll_preds, pos_label=pos_label)
    return p, r, threshold

def get_AUC(p, r):
    return (auc(r, p)*100)

## Experiments

In [18]:
arr_false = []
arr_false_h = []
arr_true = []

for v in human_label_detect_False.values():
    arr_false.extend(v)
for v in human_label_detect_False_h.values():
    arr_false_h.extend(v)
for v in human_label_detect_True.values():
    arr_true.extend(v)
    
random_baseline_false = np.mean(arr_false)
random_baseline_false_h = np.mean(arr_false_h)
random_baseline_true = np.mean(arr_true)

print("Random baseline false: ", random_baseline_false)
print("Random baseline false h: ", random_baseline_false_h)
print("Random baseline true: ", random_baseline_true)

Random baseline false:  0.7295597484276729
Random baseline false h:  0.2971813725490196
Random baseline true:  0.27044025157232704


In [19]:
def sample(scores, labels, oneminus_pred=False):
    prec, rec, threshold = get_PR_with_human_labels(scores, labels, pos_label=1, oneminus_pred=oneminus_pred)
    return np.round(get_AUC(prec, rec), 2)

def addToResult(df, name, scores):
    df = df._append({"n-gram": name, 
                  "NoFac": sample(scores, human_label_detect_False), 
                  "NoFac-H": sample(scores, human_label_detect_False_h), 
                  "Fact": sample(scores, human_label_detect_True, oneminus_pred=True)}, ignore_index = True)
    return df

def addNgramModuleToResult(df, name, scores):
    df = addToResult(df, name+"-Sent Avg", scores[0])
    df = addToResult(df, name+"-Sent Max", scores[1])
    return df

In [23]:
import pandas as pd

df = pd.DataFrame(columns=["n-gram", "NoFac", "NoFac-H", "Fact"])
df = addNgramModuleToResult(df, "Unigram", scores_unigram)
df = addNgramModuleToResult(df, "Bigram", scores_bigram)
df = addNgramModuleToResult(df, "Trigram", scores_trigram)
df = addNgramModuleToResult(df, "Four-gram", scores_fourgram)
df = addNgramModuleToResult(df, "Five-gram", scores_fivegram)
df.head(10)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_169680\2192373188.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = df._append({"n-gram": name,


,n-gram,NoFac,NoFac-H,Fact
0,Unigram-Sent Avg,81.52,40.33,41.78
1,Unigram-Sent Max,85.64,41.05,58.44
2,Bigram-Sent Avg,82.87,43.90,51.92
3,Bigram-Sent Max,85.07,38.86,58.17
4,Trigram-Sent Avg,83.74,44.34,53.42
5,Trigram-Sent Max,84.53,36.31,57.00
6,Four-gram-Sent Avg,83.81,42.92,53.65
7,Four-gram-Sent Max,83.76,35.43,55.68
8,Five-gram-Sent Avg,83.44,41.91,53.33
9,Five-gram-Sent Max,83.40,35.26,54.62
